## Import dependency

In [1]:
import pandas as pd

import json

import re
import html
from bs4 import BeautifulSoup

## Import Data

In [2]:
file_path = "../data/servicedesk_dptsi-may2025.csv"

In [3]:
df = pd.read_csv(file_path,encoding='windows-1252')
df

,ID,title,body,userid,assignedid,timestamp,categoryid,jenisid,status,priority,last_reply_timestamp,last_reply_userid,notes,message_id_hash,guest_email,last_reply_string,rating,ticket_date,close_ticket_date
0,60020,Password not recognised,"<p>Hello,&nbsp;</p>\n\n<p>When I try to connec...",0,12,1735673465,114,1,2,3,1736304668,12,NaN,dd57158e95bb70fffa3e32998d55381a,5999241069@student.its.ac.id,08/01/2025 9:51:08,0,01-01-2025,08-01-2025
1,60021,LUPA PASSWORD EMAILITS,<p>Permisi Disini saya ingin konfirmasi bahwa ...,22143,19003,1735693924,35,1,2,3,1735805638,19003,Tiket ini di tutup Otomatis oleh sistem karena...,0c5fd7d70f7bcf1582ceddf831ded7a7,NaN,02/01/2025 15:13:58,0,01-01-2025,NaN
2,60022,Lupa pasword e-mail ITS,"<p>Selamat pagi, saya yang beridentitas dibawa...",18594,12467,1735697383,35,2,2,3,1735871044,12467,Tiket ini di tutup Otomatis oleh sistem karena...,48ab9944afa5e054856536c286ac13de,NaN,03/01/2025 9:24:04,0,01-01-2025,NaN
3,60023,Tidak bisa login my ITS,<p>Selamat pagi perkenalkan saya mahasiswa bar...,21357,12467,1735697743,35,1,2,3,1735869789,12467,Tiket ini di tutup Otomatis oleh sistem karena...,33646e6dffefaff780ca32a3142b9500,NaN,03/01/2025 9:03:09,0,01-01-2025,NaN
4,60024,Tidak bisa login di myits,"<p>saya ingin mereset password gmail saya, dik...",22144,12467,1735699517,35,1,2,3,1735869765,12467,Tiket ini di tutup Otomatis oleh sistem karena...,06ea348c1ba74236b61dd3a7857995ed,NaN,03/01/2025 9:02:45,0,01-01-2025,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5691,65715,Lupa password integra,<p>Mohon bantuannya saya mahasiswa semester ak...,23871,0,1750230033,35,2,0,3,1750230033,0,NaN,c7edc03d4d15095a6b0a12f1bc1c6428,NaN,18/06/2025 14:00:33,0,18-06-2025,NaN
5692,65716,Pengajuan tanda tangan,<p>Pengajuan tanda tangan surat persutujuan pe...,23775,0,1750230271,107,2,0,3,1750230271,0,NaN,e77f93c1cb989208f89d9a34e1acc4d2,NaN,18/06/2025 14:04:31,0,18-06-2025,NaN
5693,65717,Ganti Nomor Telpon MyIts,<p>Nomor telpon myITS saya sudah tidak aktif d...,21863,0,1750230802,22,2,0,3,1750230802,0,NaN,a124f616e630ce84600e5566957209fb,NaN,18/06/2025 14:13:22,0,18-06-2025,NaN
5694,65718,Ganti Nomor Telpon MyIts,<p>Nomor telpon myITS saya sudah tidak aktif d...,21863,0,1750230808,22,2,0,3,1750230808,0,NaN,e4cc6fc12fcb2f8b6c7d146347ae5a60,NaN,18/06/2025 14:13:28,0,18-06-2025,NaN


## Labelling Code to Unit Name

### Unit Code to Unit Name List

In [4]:
code_to_unit = {
    # Unit Layanan SPMI
    114: "SPMI",

    # Unit Layanan PDDIKTI
    112: "PDDIKTI",

    # Unit Layanan Hukum dan Penanganan Isu Strategis (ULHPIS)
    65: "ULHPIS",
    66: "ULHPIS",
    67: "ULHPIS",
    68: "ULHPIS",
    71: "ULHPIS",
    73: "ULHPIS",
    74: "ULHPIS",
    76: "ULHPIS",
    80: "ULHPIS",
    84: "ULHPIS",
    87: "ULHPIS",
    89: "ULHPIS",

    # Unit Komunikasi Publik (UKP)
    156: "UKP",
    157: "UKP",
    158: "UKP",
    159: "UKP",
    160: "UKP",
    161: "UKP",
    162: "UKP",
    163: "UKP",
    164: "UKP",
    165: "UKP",
    166: "UKP",
    167: "UKP",

    # Layanan SIPMABA
    171: "SIPMABA",

    # Layanan Aplikasi e-Aspirasi
    40: "e-Aspirasi",
    41: "e-Aspirasi",

    # Kantor Penjaminan Mutu
    103: "KPM",

    # Jaring Aspirasi Terpadu
    169: "Jaring Aspirasi Terpadu",

    # Direktorat Sumber Daya Manusia dan Organisasi (DSDMO)
    13: "DSDMO",
    14: "DSDMO",
    16: "DSDMO",
    17: "DSDMO",
    18: "DSDMO",
    19: "DSDMO",
    20: "DSDMO",
    26: "DSDMO",
    31: "DSDMO",
    117: "DSDMO",
    118: "DSDMO",
    119: "DSDMO",
    127: "DSDMO",
    128: "DSDMO",
    150: "DSDMO",
    178: "DSDMO",

    # Direktorat Riset dan Pengabdian kepada Masyarakat (DRPM)
    43: "DRPM",
    58: "DRPM",
    59: "DRPM",
    149: "DRPM",

    # Direktorat Perencanaan Anggaran dan Logistik (DIPAL)
    28: "DIPAL",
    47: "DIPAL",

    # Direktorat Pengembangan Teknologi dan Sistem Informasi (DPTSI)
    22: "DPTSI",
    23: "DPTSI",
    35: "DPTSI",
    170: "DPTSI",

    # Direktorat Pendidikan
    49: "Direktorat Pendidikan",
    104: "Direktorat Pendidikan",

    # Direktorat Pascasarjana dan Pengembangan Akademik
    98: "Direktorat Pascasarjana dan Pengembangan Akademik",

    # Direktorat Kemahasiswaan (Ditmawa)
    50: "Direktorat Kemahasiswaan (Ditmawa)",
    100: "Direktorat Kemahasiswaan (Ditmawa)",
    151: "Direktorat Kemahasiswaan (Ditmawa)",

    # Biro Umum Keamanan dan K3L
    90: "Biro Umum Keamanan dan K3L",
    91: "Biro Umum Keamanan dan K3L",
    92: "Biro Umum Keamanan dan K3L",
    93: "Biro Umum Keamanan dan K3L",
    172: "Biro Umum Keamanan dan K3L",

    # Biro Keuangan
    130: "Biro Keuangan",
    132: "Biro Keuangan",
    134: "Biro Keuangan",
    135: "Biro Keuangan",
    136: "Biro Keuangan",

    # Asrama
    32: "Asrama"
}

In [5]:
df["unit_kerja"] = df["categoryid"].map(code_to_unit)
df.unit_kerja.value_counts()

unit_kerja
DPTSI                                                3307
UKP                                                   325
DSDMO                                                 245
SIPMABA                                               229
PDDIKTI                                               177
Direktorat Pendidikan                                 160
Direktorat Pascasarjana dan Pengembangan Akademik      67
Biro Keuangan                                          48
Biro Umum Keamanan dan K3L                             38
Direktorat Kemahasiswaan (Ditmawa)                     35
DRPM                                                   21
Asrama                                                 13
SPMI                                                    7
Jaring Aspirasi Terpadu                                 5
KPM                                                     2
DIPAL                                                   2
Name: count, dtype: int64

Drop the data with NaN value on Unit Kerja. 

The ticket with NaN value on Unit Kerja are belong to Department Level

In [ ]:
print("The data that have NaN value on the Unit Kerja column")
print("Before NaN drop :", len(df))
prev_len = len(df)

df.dropna(subset=['unit_kerja'], inplace=True)
df.reset_index(drop=True, inplace=True)


print("After NaN drop", len(df))
print("Dropped data : ", (prev_len-len(df)))

The data that have NaN value ont the Unit Kerja column, it direct Department
Before NaN drop : 5696
After NaN drop 4681
Dropped data :  1015


### Preprocess Text Data

In [12]:
def preprocess(text):
    # 1. Decode entitas
    text = html.unescape(text)
    
    # 2. Hapus tag HTML
    text = BeautifulSoup(text, "html.parser").get_text(separator=" ")
    
    # 3. Lowercase
    text = text.lower()
    
    # 4. Hapus punctuation (tanda baca), termasuk underscore
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    
    # 5. Normalisasi whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [13]:
df['preprocessed_text'] = (
    df['title'].fillna('') + ' ' + df['body'].fillna('')
).apply(preprocess)

## Exploratory Insight Before Applying KB

### Insight 1 : Email empty and various format

In [7]:
# Helper function to determine if a local part is purely numeric
def is_numeric_local_part(local_part):
    return bool(re.fullmatch(r'\d+', local_part))

# Helper function to determine if a local part looks like a name (contains letters, not purely numeric)
def is_name_like_local_part(local_part):
    return bool(re.search(r'[a-zA-Z]', local_part)) and not is_numeric_local_part(local_part)

# Define ITS student domains explicitly
its_student_domains = {'student.its.ac.id', 'mhs.its.ac.id',}
# Define general ITS domain
its_general_domain = 'its.ac.id'

def classify_email_category(email):
    if pd.isna(email):
        return 'Empty'

    match = re.match(r'([^@]+)@(.+)', email)
    if match:
        local_part, domain = match.groups()
        local_part_lower = local_part.lower()
        domain_lower = domain.lower()

        # Student Rule: Local part is numeric AND domain is an ITS student domain OR general its.ac.id
        if is_numeric_local_part(local_part_lower) and (domain_lower in its_student_domains or domain_lower == its_general_domain):
            return 'Student'
        # Staff Rule: Local part is name-like AND domain is general its.ac.id
        elif is_name_like_local_part(local_part_lower) and domain_lower == its_general_domain:
            return 'Staff'
        else:
            return 'Other'
    else:
        # Malformed email
        return 'Other'

# Apply the classification to create a new column
df['email_category'] = df['guest_email'].apply(classify_email_category)

# Calculate total number of rows
total_emails = len(df)

# Count the categories from the new column
empty_emails_count = (df['email_category'] == 'Empty').sum()
student_emails_count = (df['email_category'] == 'Student').sum()
staff_emails_count = (df['email_category'] == 'Staff').sum()
other_emails_count = (df['email_category'] == 'Other').sum()


print(f"Total entries: {total_emails}")
print(f"Empty emails: {empty_emails_count}")
print(f"Student emails (numeric local part, ITS domain): {student_emails_count}")
print(f"Campus staff emails (name-like local part, its.ac.id domain): {staff_emails_count}")
print(f"Other emails (emails not fitting student/staff criteria): {other_emails_count}")

Total entries: 4681
Empty emails: 2980
Student emails (numeric local part, ITS domain): 1251
Campus staff emails (name-like local part, its.ac.id domain): 376
Other emails (emails not fitting student/staff criteria): 74


In [8]:
df.guest_email[df.email_category == "Other" ]

155            03111942000012@mahasiswa.integra.its.ac.id
467                         5999241047@student.its.ac.id.
578                                     rini@is.its.ac.id
677                         6022221015@student.its.ac.id/
696                         5030241053@.student.its.ac.id
                              ...                        
4434                                 arbaati@oe.its.ac.id
4455    2035221059@student.its.ac.id / radityaafritama...
4534                               5045221008@myits.ac.id
4643                        5018231072@students.its.ac.id
4644                        5018231072@students.its.ac.id
Name: guest_email, Length: 74, dtype: str

Action : Previously, the knowledge base was planned to have role section to identify which one is lecturer or academic staff and students. But due to various email format and a lot of ticket that have empty email, the role section would not be added to the knowledge base

### Insight 2 : A lot of data contain the same values on column body  

#### Using Text before preprocessed

In [ ]:
df_with_duplicated_bodies = df[df.duplicated(subset=['body'], keep=False)]

# Calculate the counts for each duplicated body content
duplicated_body_summary_df = df_with_duplicated_bodies['body'].value_counts().to_frame(name='total_body_occurrences')
duplicated_body_summary_df.index.name = 'body_content'

# Identify rows where both 'body' and 'title' are duplicated
# This means the exact (body, title) pair appears more than once.
df_with_duplicated_body_title = df[df.duplicated(subset=['body', 'title'], keep=False)]

# Count, for each body, how many of its occurrences are part of a duplicated (body, title) pair
occurrences_with_duplicated_title_per_body = df_with_duplicated_body_title['body'].value_counts()

# Add a new column to duplicated_body_summary_df for the number of times
# a body appears with a duplicated title
duplicated_body_summary_df['occurrences_with_duplicated_title_for_body'] = \
    duplicated_body_summary_df.index.map(occurrences_with_duplicated_title_per_body).fillna(0).astype(int)

# Calculate the percentage
duplicated_body_summary_df['percentage_duplicated_title_for_body'] = (
    duplicated_body_summary_df['occurrences_with_duplicated_title_for_body'] / duplicated_body_summary_df['total_body_occurrences']
) * 100

print("Analysis of duplicated body content, including percentage of occurrences with duplicated titles:\n")
print(duplicated_body_summary_df)

Analysis of duplicated body content, including percentage of occurrences with duplicated titles:

                                                    total_body_occurrences  \
body_content                                                                 
<p>Ketika ingin membuka MyITS, saya diminta unt...                      10   
<p>selamat pagi, mohon maaf mengganggu waktunya...                       8   
<p>Selamat Pagi</p>\n\n<p>Mohon bantuan untuk l...                       5   
<p>Kepada, Bapak/Ibu Pejabat Pengelolan Layanan...                       5   
<p>Yth Admin DPTSI</p>\n\n<p>Mohon bantuannya u...                       4   
...                                                                    ...   
<p>Otentikasi saya bermasalah. Web meminta kode...                       2   
<p>Yth. Direktorat Pengembangan Teknolofi dan S...                       2   
<p>Tidak dapat login pada myits single sing-on ...                       2   
<p>Selamat Pagi,</p>\n\n<p>Perkenalkan saya 

In [10]:
duplicated_body_summary_df.sort_values(by="percentage_duplicated_title_for_body")[duplicated_body_summary_df.percentage_duplicated_title_for_body < 50]

C:\Users\Asus\AppData\Local\Temp\ipykernel_26344\1297792913.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  duplicated_body_summary_df.sort_values(by="percentage_duplicated_title_for_body")[duplicated_body_summary_df.percentage_duplicated_title_for_body < 50]


,total_body_occurrences,occurrences_with_duplicated_title_for_body,percentage_duplicated_title_for_body
body_content,,,
"<p>Caption :</p>\n\n<p>[SEMANISKALA: YANG TERINGAT, TERKENANG, DAN TERIKAT]</p>\n\n<p>&nbsp;</p>\n\n<p>Semaniskala sebagai penutup dari seluruh rangkaian acara ArchProject 2025, disampaikan melalui ruang kreatif yang menjadi media bernostalgia bagi kalian, Sang Penjelajah Waktu!</p>\n\n<p>&nbsp;</p>\n\n<p>",3,0,0.0
"<p>Untuk memperjelas urusan administrasi Seleksi Calon Pegawai Negeri Sipil 2024, saya memohon untuk dibuatkan Surat Pernyataan yang menjelaskan terkait perbedaan Departemen Teknik Perkapalan dan Teknik Sistem Perkapalan</p>",2,0,0.0
"<p>Nama: Pramana Tabah Pangestu&nbsp;</p>\n\n<p>Departemen: Teknik Elektro Otomasi</p>\n\n<p>NRP: 2041221024</p>\n\n<p>Permisi, disini saya ingin mengajukan Layanan Liputan Internal dari pihak media ITSTV untuk meliput kegiatan IARC ITS departemen Teknik Elektro Otomasi.</p>\n\n<p>Kegiatan IARC ini diadakan di BG Junction Surabaya untuk lomba robot (LTA, LTM, Soccer) dan di Departemen Teknik Elektro Otomasi untuk lomba PLC dan LKTI yg semuanya kategori SMA/SMK/Sederajat.</p>\n\n<p>Acara ini diselenggarakan pada :<br />\nHari Minggu&nbsp;tanggal 12 Januari&nbsp;2023 pukul 08.30&nbsp;- 17.50</p>\n\n<p>Oleh karena itu,saya mohon untuk bersedia pada jadwal kami.</p>\n\n<p>Terimakasih</p>",2,0,0.0
<p>Padahal survey SAR belum saya&nbsp;isi. Tp sistem nya nge bug. Tiba-tiba SAR saya sudah permanen. Sekarang&nbsp;akibatnya&nbsp;saya&nbsp;jadi tidak bisa&nbsp;permanen nilai..tombol permanen nilai di MyITS academics utk mata kuliah tersebut&nbsp;tidak muncul</p>,2,0,0.0
<p>Mahasiswa kami S2 PJJ REB - Teknik Elektro tidak bisa login myITS SSO:<br />\nEmail mhs : 6022241031@student.its.ac.id<br />\nNew Pass : T@mi1993<br />\n<br />\nMohon bantuan</p>,2,0,0.0
<p>Email ITS saya belum bisa diakses karena setiap ingin log in selalu salah password dan tulisannya belum di registrasi. kemarin sudah mengganti passwordnya dan kemarin diberikan data seperti:</p>\n\n<p>Nama : Dwi Rahmat Septiandi</p>\n\n<p>Departemen : Teknik Mesin</p>\n\n<p>NRP : 5007211172</p>\n\n<p>Email ITS : 5007211172@student.its.ac.id</p>\n\n<p>Email cadangan : rahmatseptian49@gmail.com</p>\n\n<p>Dilampirkan scan KTP</p>,2,0,0.0
<p>lupa password my its</p>,2,0,0.0
"<p>Halo selamat siang Bapak/Ibu yang terhormat, saya Ikhsan Hafizh Nirwasita dengan NRP 5003231200 dari departemen Statistika. Saya kesulitan untuk mengakses akun saya, dulunya saya hanya tau mengakses akun ITS menggunakan NRP dan password saja tetapi sekarang menggunakan email ITS dan password. Saya sudah memasukkan password saya sebelumnya dan harus mengatur ulang password&nbsp;barkali-kali dari minggu lalu. Mohon bantuannya Bapak/Ibu, saya sangat urgent sekali. Saya lampirkan nomor handphone saya yakni 085780413914 dan email pribadi saya ikhsanhn1409@gmail.com.</p>\n\n<p>Atas perhatian Bapak/Ibu, saya ucapkan terima kasih.</p>\n\n<p>TTD</p>\n\n<p>Ikhsan Hafizh Nirwasita, 5003231200</p>",2,0,0.0
"<p>Desa Kalanganyar, Sedati, Sidoarjo, terkenal dengan produk olahan bandeng-nya, menghadapi penurunan potensi akibat adanya banjir rob yang merusak tambak desa. Masalah ini mendorong tim Sobi ITS yaitu Penerima Beasiswa Sobat Bumi Institut Teknologi Sepuluh Nopember untuk berkontribusi melalui program &amp;ldquo;Aksi Sobat Bumi&amp;rdquo; dengan memulai aksi lingkungan dengan menanam 100 pohon trembesi dan mengedukasi masyarakat tentang pentingnya keberlanjutan lingkungan pada 24 Desember 2024 lalu. Langkah ini kemudian dirumuskan dalam gerakan &amp;ldquo;Green Activ-ITS&amp;rdquo; yang bertujuan untuk menginspirasi aksi hijau yang inovatif, aktif, dan berkelanjutan. Melalui kolaborasi dengan pemerintah desa dan karang taruna, program ini diharapkan memberikan manfaat nyata bagi Desa Kalanganyar dan menginspirasi masyarakat Jawa Timur, khususnya dalam mengatasi tantangan lingkungan hidup dengan solusi yang berkelanjutan.</p>",2,0,0.0


there are some data that have different title, through they have the same and exact body. 
* question for supervisor : if it still have to be distinct, which title that suitable to be choosed?
* suggestion : distinct for those that have duplicated title and body


#### Using Teks After Preprocessed

In [27]:
df_with_duplicated_bodies_preprocessed = df[df.duplicated(subset=['preprocessed_text'], keep=False)]

print(len(df_with_duplicated_bodies_preprocessed))
# # Calculate the counts for each duplicated body content
# duplicated_body_summary_df_preprocessed = df_with_duplicated_bodies_preprocessed['preprocessed_text'].value_counts().to_frame(name='total_content_occurrences')
# duplicated_body_summary_df_preprocessed.index.name = 'text_content'

# print("Analysis of duplicated Content:\n")
# print(duplicated_body_summary_df_preprocessed)

413


In [23]:
df_with_duplicated_bodies_preprocessed

,ID,title,body,userid,assignedid,timestamp,categoryid,jenisid,status,priority,...,notes,message_id_hash,guest_email,last_reply_string,rating,ticket_date,close_ticket_date,unit_kerja,email_category,preprocessed_text
23,60044,Lupa password email,<p>Mohon bantuannya untuk meresetkan password</p>,0,19002,1735726934,35,2,2,1,...,Tiket ini di tutup Otomatis oleh sistem karena...,69d2e58038fc82ae1d89062e0134ea7f,azamiriziq@its.ac.id,03/01/2025 8:24:38,0,01-01-2025,NaN,DPTSI,Staff,lupa password email mohon bantuannya untuk mer...
24,60045,Lupa password email,<p>Mohon bantuannya untuk meresetkan password</p>,0,19002,1735727341,35,2,2,1,...,Tiket ini di tutup Otomatis oleh sistem karena...,f508cb5b6f1f5874e9353f8732ef2501,azamiriziq@its.ac.id,03/01/2025 8:27:11,0,01-01-2025,NaN,DPTSI,Staff,lupa password email mohon bantuannya untuk mer...
33,60057,Problem di iThenticate License,"<p>Selamat pagi, maaf izin bertanya dan tolong...",22159,0,1735767774,22,1,2,3,...,<p>DUPLIKAT [60058]</p>,3c394f66a5fbf6fe5454cdd3c186c404,NaN,02/01/2025 4:42:54,0,02-01-2025,03-01-2025,DPTSI,Empty,problem di ithenticate license selamat pagi ma...
34,60058,Problem di iThenticate License,"<p>Selamat pagi, maaf izin bertanya dan tolong...",22159,0,1735767828,22,1,2,3,...,<p>DUPLIKAT [60090]</p>,f3dd15f5db76bdd749066b349bea36d1,NaN,02/01/2025 4:43:48,0,02-01-2025,03-01-2025,DPTSI,Empty,problem di ithenticate license selamat pagi ma...
49,60073,Konversi Mata Kuliah,<p>Saya telah mengikuti kegiatan magang di sem...,0,12617,1735783999,35,1,2,3,...,Tiket ini di tutup Otomatis oleh sistem karena...,691f38e8256a63fe6c41dc2a1cad4b17,5015211148@student.its.ac.id,10/01/2025 13:24:42,0,02-01-2025,NaN,DPTSI,Student,konversi mata kuliah saya telah mengikuti kegi...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4649,65680,Tidak dapat login myits,<p>Tidak dapat login pada myits single sing-on...,9845,0,1750207232,23,2,2,3,...,<p>Double Ticket&nbsp;65679</p>,1a3d7ee61e502e52ee23a6ed44bf0b6f,NaN,18/06/2025 07:40:32,0,18-06-2025,18-06-2025,DPTSI,Empty,tidak dapat login myits tidak dapat login pada...
4656,65688,Kendala Tidak Bisa Masuk MyITS Portal,"<p>Selamat Pagi,</p>\n\n<p>Perkenalkan saya Di...",0,19002,1750210842,35,1,1,3,...,NaN,8bece7573c60efd668282328cb52e6b0,5011221132@student.its.ac.id,18/06/2025 10:53:44,0,18-06-2025,NaN,DPTSI,Student,kendala tidak bisa masuk myits portal selamat ...
4657,65689,Kendala Tidak Bisa Masuk MyITS Portal,"<p>Selamat Pagi,</p>\n\n<p>Perkenalkan saya Di...",0,19002,1750211432,35,1,1,3,...,NaN,01cf131148ac47195353c6a14f3b1ad9,5011221132@student.its.ac.id,18/06/2025 13:29:31,0,18-06-2025,NaN,DPTSI,Student,kendala tidak bisa masuk myits portal selamat ...
4679,65717,Ganti Nomor Telpon MyIts,<p>Nomor telpon myITS saya sudah tidak aktif d...,21863,0,1750230802,22,2,0,3,...,NaN,a124f616e630ce84600e5566957209fb,NaN,18/06/2025 14:13:22,0,18-06-2025,NaN,DPTSI,Empty,ganti nomor telpon myits nomor telpon myits sa...


In [31]:
print("Duplicated removal impact would take the unique value from",len(df_with_duplicated_bodies_preprocessed), "rows to", len(duplicated_body_summary_df_preprocessed), "rows")
print("It will removes",(len(df_with_duplicated_bodies_preprocessed)-len(duplicated_body_summary_df_preprocessed)), "rows")

Duplicated removal impact would take the unique value from 413 rows to 182 rows
It will removes 231 rows


In [ ]:
print("The data that have duplicated body and title after being preprocessed")
print("Before drop duplicates :", len(df))
prev_len = len(df)

df = df.drop_duplicates(subset=['preprocessed_text'], keep='first')
df.reset_index(drop=True, inplace=True)

print("After drop duplicates :", len(df))
print("Dropped data : ", (prev_len-len(df)))

The data that have duplicated body and title after being preprocessed
Before drop duplicates : 4681
After drop duplicates : 4450
Dropped data :  231


## Implement Knowledge Base

### First Implementation

call knowledge base 

In [32]:
def load_kb(file_path):
    with open(file_path, 'r') as f:
        return json.load(f)

In [33]:
def preprocess_kb(kb):
    processed_kb = {}
    for unit, levels in kb.items():
        processed_kb[unit] = {}
        for level, items in levels.items():
            if not isinstance(items, list):   # ← ADD THIS LINE
                continue                       # ← ADD THIS LINE
            processed_items = []
            for item in items:
                processed_item = {
                    "canonical": item["canonical"],
                    "canonical_clean": preprocess(item["canonical"]),
                    "variants_clean": [preprocess(v) for v in item.get("variants", [])],
                }
                processed_items.append(processed_item)
            processed_kb[unit][level] = processed_items
    return processed_kb


In [13]:
kb_data = load_kb('../utils/kb_20260408_first.json')

In [34]:
def implement_kb(row, kb_raw):
    # Use pre-cleaned text from the DataFrame columns (no preprocessing here)
    full_text = row['preprocessed_text']
    kb = preprocess_kb(kb_raw)

    WEIGHT_MAP = {"high": 3, "low": 1}
    normalized_scores = {}
    raw_scores = {}
    match_details = {}

    # --- STEP 1: OVERRIDE CHECK (Skor Otomatis 1.0) ---
    for unit, levels in kb.items():
        if "override" in levels:
            for item in levels["override"]:
                keywords = [item["canonical_clean"]] + item.get("variants_clean", [])

                if any(re.search(rf"\b{re.escape(kw)}\b", full_text) for kw in keywords if kw):
                    return pd.Series([unit, 1.0, 1.0, [item["canonical"]], f"{unit}: Override", f"{unit}: {item['canonical']}"])

    # --- STEP 2: WEIGHTED SCORING (High & Low) ---
    for unit, levels in kb.items():
        unit_raw_score = 0
        total_possible_weight = 0
        matches = []

        for level, items in levels.items():
            if level == "override":
                continue

            weight = WEIGHT_MAP.get(level, 0)
            for item in items:
                keywords = [item["canonical_clean"]] + item.get("variants_clean", [])

                for kw in keywords:
                    if not kw: continue

                    total_possible_weight += weight
                    if re.search(rf"\b{re.escape(kw)}\b", full_text):
                        unit_raw_score += weight
                        matches.append(kw)

        if matches:
            normalized_scores[unit] = unit_raw_score / total_possible_weight if total_possible_weight > 0 else 0
            raw_scores[unit] = unit_raw_score
            match_details[unit] = list(set(matches))

    # --- STEP 3: SELEKSI TERBAIK ---
    if not normalized_scores:
        return pd.Series(["UNKNOWN", 0.0, 0.0, [], "", ""])

    sorted_units = sorted(normalized_scores.items(), key=lambda x: x[1], reverse=True)
    best_unit, best_norm_score = sorted_units[0]

    second_norm_score = sorted_units[1][1] if len(sorted_units) > 1 else 0
    confidence = best_norm_score / (best_norm_score + second_norm_score) if (best_norm_score + second_norm_score) > 0 else 1.0

    all_scores_str = ", ".join([f"{u}: {raw_scores[u]}" for u in raw_scores])
    all_keywords_str = ", ".join([f"{u}: {'|'.join(match_details[u])}" for u in match_details])
    
    return pd.Series([
        best_unit,
        round(best_norm_score, 3),
        round(confidence, 3),
        match_details.get(best_unit, []),
        all_scores_str,
        all_keywords_str
    ])

In [38]:
df_kb_label = df.copy()

In [20]:
df_kb_label[['predicted_unit', 'score', 'confidence', 'keywords', 'all_unit_scores', 'all_unit_keywords']] = df.apply(
    lambda row: implement_kb(row, kb_data), axis=1
)

In [18]:
df_kb_label.to_excel("../data/output/20260408_kb_implement_match_keyword.xlsx", index=False)

### Iteration 1

In [19]:
df_kb_label_1 = df_kb_label.copy()

In [20]:
kb_iteration_1 = load_kb("../utils/kb_20260409_iteration_1.json")

In [21]:
df_kb_label_1[['predicted_unit', 'score', 'confidence', 'keywords', 'all_unit_scores', 'all_unit_keywords']] = df.apply(
    lambda row: implement_kb(row, kb_iteration_1), axis=1
)

df_kb_label_1.to_excel("../data/output/20260410_kb_implement_match_keyword_iteration_1.xlsx", index=False)

In [22]:
df_kb_label_1['predicted_unit'].value_counts()

predicted_unit
DPTSI (Direktorat Pengembangan Teknologi dan Sistem Informasi)                 2576
UNKNOWN                                                                         901
DPSP (Direktorat Pendidikan Sarjana dan Pascasarjana)                           377
DITMAWA (Direktorat Kemahasiswaan)                                              215
DSDMO (Direktorat Sumber Manusia dan Organisasi                                 199
UKP (Unit Komunikasi Publik)                                                    139
UNCLASSIFIED                                                                    130
BK (Biro Keuangan)                                                               57
DPPS (Direktorat Perencanaan dan Pengembangan Strategis)                         33
DRPM (Direktorat Riset dan Pengabdian Masyarakat)                                30
BUK4L (Biro Umum dan Keamanan, Keselamatan, Kesehatan Kerja dan Lingkungan)      11
ULH (Unit Layanan Hukum)                                     

In [23]:
df_kb_label_1

,ID,title,body,userid,assignedid,timestamp,categoryid,jenisid,status,priority,...,close_ticket_date,unit_kerja,email_category,preprocessed_text,predicted_unit,score,confidence,keywords,all_unit_scores,all_unit_keywords
0,60020,Password not recognised,"<p>Hello,&nbsp;</p>\n\n<p>When I try to connec...",0,12,1735673465,114,1,2,3,...,08-01-2025,SPMI,Student,password not recognised hello when i try to co...,DPTSI (Direktorat Pengembangan Teknologi dan S...,1.0,1.0,[password issue],DPTSI (Direktorat Pengembangan Teknologi dan S...,DPTSI (Direktorat Pengembangan Teknologi dan S...
1,60021,LUPA PASSWORD EMAILITS,<p>Permisi Disini saya ingin konfirmasi bahwa ...,22143,19003,1735693924,35,1,2,3,...,NaN,DPTSI,Empty,lupa password emailits permisi disini saya ing...,DPTSI (Direktorat Pengembangan Teknologi dan S...,1.0,1.0,[password issue],DPTSI (Direktorat Pengembangan Teknologi dan S...,DPTSI (Direktorat Pengembangan Teknologi dan S...
2,60022,Lupa pasword e-mail ITS,"<p>Selamat pagi, saya yang beridentitas dibawa...",18594,12467,1735697383,35,2,2,3,...,NaN,DPTSI,Empty,lupa pasword e mail its selamat pagi saya yang...,DPTSI (Direktorat Pengembangan Teknologi dan S...,1.0,1.0,[password issue],DPTSI (Direktorat Pengembangan Teknologi dan S...,DPTSI (Direktorat Pengembangan Teknologi dan S...
3,60023,Tidak bisa login my ITS,<p>Selamat pagi perkenalkan saya mahasiswa bar...,21357,12467,1735697743,35,1,2,3,...,NaN,DPTSI,Empty,tidak bisa login my its selamat pagi perkenalk...,DPTSI (Direktorat Pengembangan Teknologi dan S...,1.0,1.0,[password issue],DPTSI (Direktorat Pengembangan Teknologi dan S...,DPTSI (Direktorat Pengembangan Teknologi dan S...
4,60024,Tidak bisa login di myits,"<p>saya ingin mereset password gmail saya, dik...",22144,12467,1735699517,35,1,2,3,...,NaN,DPTSI,Empty,tidak bisa login di myits saya ingin mereset p...,DPTSI (Direktorat Pengembangan Teknologi dan S...,1.0,1.0,[password issue],DPTSI (Direktorat Pengembangan Teknologi dan S...,DPTSI (Direktorat Pengembangan Teknologi dan S...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4676,65711,Akun myits tidak bisa terutentikasi,<p>aplikasi authenticator tidak memunculkan ko...,23870,19002,1750226938,35,1,1,3,...,NaN,DPTSI,Empty,akun myits tidak bisa terutentikasi aplikasi a...,DPTSI (Direktorat Pengembangan Teknologi dan S...,1.0,1.0,[akun its],DPTSI (Direktorat Pengembangan Teknologi dan S...,DPTSI (Direktorat Pengembangan Teknologi dan S...
4677,65714,"Tidak bisa login myits, karena apk authenticat...","<p>Tidak bisa login myits, karena apk authenti...",23872,0,1750230015,35,2,0,3,...,NaN,DPTSI,Empty,tidak bisa login myits karena apk authenticato...,DPTSI (Direktorat Pengembangan Teknologi dan S...,1.0,1.0,[password issue],DPTSI (Direktorat Pengembangan Teknologi dan S...,DPTSI (Direktorat Pengembangan Teknologi dan S...
4678,65715,Lupa password integra,<p>Mohon bantuannya saya mahasiswa semester ak...,23871,0,1750230033,35,2,0,3,...,NaN,DPTSI,Empty,lupa password integra mohon bantuannya saya ma...,DPTSI (Direktorat Pengembangan Teknologi dan S...,1.0,1.0,[password issue],DPTSI (Direktorat Pengembangan Teknologi dan S...,DPTSI (Direktorat Pengembangan Teknologi dan S...
4679,65717,Ganti Nomor Telpon MyIts,<p>Nomor telpon myITS saya sudah tidak aktif d...,21863,0,1750230802,22,2,0,3,...,NaN,DPTSI,Empty,ganti nomor telpon myits nomor telpon myits sa...,DPTSI (Direktorat Pengembangan Teknologi dan S...,1.0,1.0,[password issue],DPTSI (Direktorat Pengembangan Teknologi dan S...,DPTSI (Direktorat Pengembangan Teknologi dan S...


### Iteration 2

In [24]:
df_kb_label_2 = df_kb_label.copy()

In [25]:
kb_iteration_2 = load_kb("../utils/kb_20260414_iteration_2.json")

In [26]:
df_kb_label_2[['predicted_unit', 'score', 'confidence', 'keywords', 'all_unit_scores', 'all_unit_keywords']] = df.apply(
    lambda row: implement_kb(row, kb_iteration_2), axis=1
)

In [27]:
df_kb_label_2['predicted_unit'].value_counts()

predicted_unit
DPTSI (Direktorat Pengembangan Teknologi dan Sistem Informasi)                 2961
DPSP (Direktorat Pendidikan Sarjana dan Pascasarjana)                           484
DSDMO (Direktorat Sumber Manusia dan Organisasi                                 314
UNCLASSIFIED                                                                    244
DITMAWA (Direktorat Kemahasiswaan)                                              241
UKP (Unit Komunikasi Publik)                                                    203
BK (Biro Keuangan)                                                               64
UNKNOWN                                                                          48
DPPS (Direktorat Perencanaan dan Pengembangan Strategis)                         47
DRPM (Direktorat Riset dan Pengabdian Masyarakat)                                30
BUK4L (Biro Umum dan Keamanan, Keselamatan, Kesehatan Kerja dan Lingkungan)      21
KPM (Kantor Penjaminan Mutu)                                 

In [28]:
df_kb_label_2

,ID,title,body,userid,assignedid,timestamp,categoryid,jenisid,status,priority,...,close_ticket_date,unit_kerja,email_category,preprocessed_text,predicted_unit,score,confidence,keywords,all_unit_scores,all_unit_keywords
0,60020,Password not recognised,"<p>Hello,&nbsp;</p>\n\n<p>When I try to connec...",0,12,1735673465,114,1,2,3,...,08-01-2025,SPMI,Student,password not recognised hello when i try to co...,DPTSI (Direktorat Pengembangan Teknologi dan S...,1.0,1.0,[password issue],DPTSI (Direktorat Pengembangan Teknologi dan S...,DPTSI (Direktorat Pengembangan Teknologi dan S...
1,60021,LUPA PASSWORD EMAILITS,<p>Permisi Disini saya ingin konfirmasi bahwa ...,22143,19003,1735693924,35,1,2,3,...,NaN,DPTSI,Empty,lupa password emailits permisi disini saya ing...,DPTSI (Direktorat Pengembangan Teknologi dan S...,1.0,1.0,[password issue],DPTSI (Direktorat Pengembangan Teknologi dan S...,DPTSI (Direktorat Pengembangan Teknologi dan S...
2,60022,Lupa pasword e-mail ITS,"<p>Selamat pagi, saya yang beridentitas dibawa...",18594,12467,1735697383,35,2,2,3,...,NaN,DPTSI,Empty,lupa pasword e mail its selamat pagi saya yang...,DPTSI (Direktorat Pengembangan Teknologi dan S...,1.0,1.0,[password issue],DPTSI (Direktorat Pengembangan Teknologi dan S...,DPTSI (Direktorat Pengembangan Teknologi dan S...
3,60023,Tidak bisa login my ITS,<p>Selamat pagi perkenalkan saya mahasiswa bar...,21357,12467,1735697743,35,1,2,3,...,NaN,DPTSI,Empty,tidak bisa login my its selamat pagi perkenalk...,DPTSI (Direktorat Pengembangan Teknologi dan S...,1.0,1.0,[password issue],DPTSI (Direktorat Pengembangan Teknologi dan S...,DPTSI (Direktorat Pengembangan Teknologi dan S...
4,60024,Tidak bisa login di myits,"<p>saya ingin mereset password gmail saya, dik...",22144,12467,1735699517,35,1,2,3,...,NaN,DPTSI,Empty,tidak bisa login di myits saya ingin mereset p...,DPTSI (Direktorat Pengembangan Teknologi dan S...,1.0,1.0,[password issue],DPTSI (Direktorat Pengembangan Teknologi dan S...,DPTSI (Direktorat Pengembangan Teknologi dan S...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4676,65711,Akun myits tidak bisa terutentikasi,<p>aplikasi authenticator tidak memunculkan ko...,23870,19002,1750226938,35,1,1,3,...,NaN,DPTSI,Empty,akun myits tidak bisa terutentikasi aplikasi a...,DPTSI (Direktorat Pengembangan Teknologi dan S...,1.0,1.0,[akun its],DPTSI (Direktorat Pengembangan Teknologi dan S...,DPTSI (Direktorat Pengembangan Teknologi dan S...
4677,65714,"Tidak bisa login myits, karena apk authenticat...","<p>Tidak bisa login myits, karena apk authenti...",23872,0,1750230015,35,2,0,3,...,NaN,DPTSI,Empty,tidak bisa login myits karena apk authenticato...,DPTSI (Direktorat Pengembangan Teknologi dan S...,1.0,1.0,[password issue],DPTSI (Direktorat Pengembangan Teknologi dan S...,DPTSI (Direktorat Pengembangan Teknologi dan S...
4678,65715,Lupa password integra,<p>Mohon bantuannya saya mahasiswa semester ak...,23871,0,1750230033,35,2,0,3,...,NaN,DPTSI,Empty,lupa password integra mohon bantuannya saya ma...,DPTSI (Direktorat Pengembangan Teknologi dan S...,1.0,1.0,[password issue],DPTSI (Direktorat Pengembangan Teknologi dan S...,DPTSI (Direktorat Pengembangan Teknologi dan S...
4679,65717,Ganti Nomor Telpon MyIts,<p>Nomor telpon myITS saya sudah tidak aktif d...,21863,0,1750230802,22,2,0,3,...,NaN,DPTSI,Empty,ganti nomor telpon myits nomor telpon myits sa...,DPTSI (Direktorat Pengembangan Teknologi dan S...,1.0,1.0,[password issue],DPTSI (Direktorat Pengembangan Teknologi dan S...,DPTSI (Direktorat Pengembangan Teknologi dan S...


In [29]:
df_kb_label_2.to_excel("../data/output/20260417_kb_implement_match_keyword_iteration_2_002.xlsx", index=False)

Output Clean Data

In [47]:
df_for_model = pd.DataFrame()
df_for_model['full_text'] = df_kb_label_2['title'].fillna('') + ' ' + df_kb_label_2['body'].fillna('')
df_for_model['clean_text'] = df_kb_label_2['preprocessed_text']
df_for_model['unit'] = df_kb_label_2['predicted_unit']

df_for_model

,full_text,clean_text,unit
0,"Password not recognised <p>Hello,&nbsp;</p>\n\...",password not recognised hello when i try to co...,DPTSI (Direktorat Pengembangan Teknologi dan S...
1,LUPA PASSWORD EMAILITS <p>Permisi Disini saya ...,lupa password emailits permisi disini saya ing...,DPTSI (Direktorat Pengembangan Teknologi dan S...
2,"Lupa pasword e-mail ITS <p>Selamat pagi, saya ...",lupa pasword e mail its selamat pagi saya yang...,DPTSI (Direktorat Pengembangan Teknologi dan S...
3,Tidak bisa login my ITS <p>Selamat pagi perken...,tidak bisa login my its selamat pagi perkenalk...,DPTSI (Direktorat Pengembangan Teknologi dan S...
4,Tidak bisa login di myits <p>saya ingin merese...,tidak bisa login di myits saya ingin mereset p...,DPTSI (Direktorat Pengembangan Teknologi dan S...
...,...,...,...
4676,Akun myits tidak bisa terutentikasi <p>aplikas...,akun myits tidak bisa terutentikasi aplikasi a...,DPTSI (Direktorat Pengembangan Teknologi dan S...
4677,"Tidak bisa login myits, karena apk authenticat...",tidak bisa login myits karena apk authenticato...,DPTSI (Direktorat Pengembangan Teknologi dan S...
4678,Lupa password integra <p>Mohon bantuannya saya...,lupa password integra mohon bantuannya saya ma...,DPTSI (Direktorat Pengembangan Teknologi dan S...
4679,Ganti Nomor Telpon MyIts <p>Nomor telpon myITS...,ganti nomor telpon myits nomor telpon myits sa...,DPTSI (Direktorat Pengembangan Teknologi dan S...


In [50]:
df_for_model.to_csv("../data/output/data_for_model.csv", index=False)

## Using Data Labeled From DPTSI

### Import Data

In [2]:
file_path_labeled = "..\data\input\pelabelan_service_desk_labeled_by_dptsi.xlsx"

In [3]:
df_labeled = pd.read_excel(file_path_labeled)
df_labeled

,ID,title,body,matched_keywords,unit_match_other,matched_keywords_other_unit,labeled_FA
0,60020,Password not recognised,"<p>Hello,&nbsp;</p>\n\n<p>When I try to connec...",password,NaN,NaN,DPTSI
1,60021,LUPA PASSWORD EMAILITS,<p>Permisi Disini saya ingin konfirmasi bahwa ...,"password, kata sandi",NaN,NaN,DPTSI
2,60022,Lupa pasword e-mail ITS,"<p>Selamat pagi, saya yang beridentitas dibawa...",NaN,NaN,NaN,DPTSI
3,60023,Tidak bisa login my ITS,<p>Selamat pagi perkenalkan saya mahasiswa bar...,NaN,NaN,NaN,DPTSI
4,60024,Tidak bisa login di myits,"<p>saya ingin mereset password gmail saya, dik...","portal, password",NaN,NaN,DPTSI
...,...,...,...,...,...,...,...
4676,65711,Akun myits tidak bisa terutentikasi,<p>aplikasi authenticator tidak memunculkan ko...,aplikasi,NaN,NaN,DPTSI
4677,65714,"Tidak bisa login myits, karena apk authenticat...","<p>Tidak bisa login myits, karena apk authenti...",NaN,NaN,NaN,DPTSI
4678,65715,Lupa password integra,<p>Mohon bantuannya saya mahasiswa semester ak...,password,NaN,NaN,DPTSI
4679,65717,Ganti Nomor Telpon MyIts,<p>Nomor telpon myITS saya sudah tidak aktif d...,NaN,NaN,NaN,DPTSI


In [5]:
df_labeled.labeled_FA.value_counts()

labeled_FA
DPTSI                                  2982
DSDMO                                   553
DPSP                                    353
BK                                      214
Ditmawa                                 180
UKP                                     159
DRPM                                    107
DIRPAIP                                  92
BMA                                      17
Departemen Teknik Elektro                 9
Departemen Teknik Sistem Perkapalan       5
Departemen Sistem Informasi               2
SDMO                                      2
PLT                                       2
Departemen Teknik Kimia                   1
Departemen Kimia                          1
UP3                                       1
BUK4L                                     1
Name: count, dtype: int64

note : 
* sdmo --> dsdmo
* plt --> buk4l

In [6]:
df_labeled['labeled_FA'] = df_labeled['labeled_FA'].replace({'SDMO': 'DSDMO', 'PLT': 'BUK4L'})

In [7]:
df_labeled['labeled_FA'].value_counts()

labeled_FA
DPTSI                                  2982
DSDMO                                   555
DPSP                                    353
BK                                      214
Ditmawa                                 180
UKP                                     159
DRPM                                    107
DIRPAIP                                  92
BMA                                      17
Departemen Teknik Elektro                 9
Departemen Teknik Sistem Perkapalan       5
BUK4L                                     3
Departemen Sistem Informasi               2
Departemen Teknik Kimia                   1
Departemen Kimia                          1
UP3                                       1
Name: count, dtype: int64

In [8]:
df_labeled.to_excel("../data/output/20260430_data_DPTSI_Labeled_fixed.xlsx", index=False)

In [8]:
departement_label = [
    "Departemen Teknik Elektro",
    "Departemen Teknik Sistem Perkapalan",
    "Departemen Sistem Informasi",
    "Departemen Teknik Kimia",
    "Departemen Kimia"
    ]

In [9]:
df_labeled[df_labeled["labeled_FA"].isin(departement_label)]

,ID,title,body,matched_keywords,unit_match_other,matched_keywords_other_unit,labeled_FA
73,60105,Pengubahan Absensi,"<p>Selamat Siang, Saya Muhammad Irfan Hanifa 5...",NaN,NaN,NaN,Departemen Teknik Kimia
99,60136,Absen Mahasiswa Joint Degree Dep Teknik Perkap...,<p>Absensi kehadiran mhs Joint Degree Dept Tek...,NaN,NaN,NaN,Departemen Teknik Sistem Perkapalan
172,60228,Permohonan Pembuatan Surat Penyataan Perbedaan...,<p>Untuk memperjelas urusan administrasi Selek...,NaN,NaN,NaN,Departemen Teknik Sistem Perkapalan
205,60269,liputan ITSTV di acara IARC ITS departemen tek...,<p>Nama: Pramana Tabah Pangestu&nbsp;</p>\n\n<...,"liputan, media",NaN,NaN,Departemen Teknik Elektro
214,60281,Permohonan Pembuatan Surat Pernyataan Perbedaa...,<p>Untuk memperjelas urusan administrasi Selek...,NaN,NaN,NaN,Departemen Teknik Sistem Perkapalan
369,60466,Permohonan Penyediaan Jaringan WiFi 2.4 GHz un...,<p><strong>Kepada Yth.</strong><br />\nKepala ...,"wifi, jaringan",NaN,NaN,Departemen Sistem Informasi
501,60623,Keluhan login akun UPBG,"<p>Akun UPBG saya lupa menggunakan email apa, ...",NaN,NaN,NaN,Departemen Teknik Sistem Perkapalan
502,60624,Permohonan akses manager classroom,"<p>Selamat pagi,</p>\n\n<p>Mohon bantunnya unt...",data,NaN,NaN,Departemen Teknik Elektro
552,60687,Permohonan Penyesuaian Jaringan WiFi dengan Ba...,<p>Kepada Yth.<br />\nDirektur Pengembangan Te...,"wifi, jaringan",NaN,NaN,Departemen Sistem Informasi
568,60705,Akun My ITS Akademik dosen menu tidak muncul,"<p>Selamat Pagi, Saya Rizal Fani dari departem...",NaN,NaN,NaN,Departemen Kimia


In [10]:
df_labeled[df_labeled.ID == 60624].body

502    <p>Selamat pagi,</p>\n\n<p>Mohon bantunnya unt...
Name: body, dtype: str

In [11]:
df_labeled[df_labeled.labeled_FA == "UP3"]

,ID,title,body,matched_keywords,unit_match_other,matched_keywords_other_unit,labeled_FA
974,61197,Akses akun Kepala Departemen DKV di myITS Kinerja,<p>Rekan-rekan DPTSI Ysh.<br />\nSaya Rahmatsy...,myits kinerja,NaN,NaN,UP3


### Iteration 3

In [66]:
df_kb_label_3 = df_kb_label.copy()

In [67]:
kb_iteration_3 = load_kb("../utils/kb_20260427_iteration_3.json")

In [68]:
df_kb_label_3

,ID,title,body,userid,assignedid,timestamp,categoryid,jenisid,status,priority,...,last_reply_userid,notes,message_id_hash,guest_email,last_reply_string,rating,ticket_date,close_ticket_date,unit_kerja,preprocessed_text
0,60020,Password not recognised,"<p>Hello,&nbsp;</p>\n\n<p>When I try to connec...",0,12,1735673465,114,1,2,3,...,12,NaN,dd57158e95bb70fffa3e32998d55381a,5999241069@student.its.ac.id,08/01/2025 9:51:08,0,01-01-2025,08-01-2025,SPMI,password not recognised hello when i try to co...
1,60021,LUPA PASSWORD EMAILITS,<p>Permisi Disini saya ingin konfirmasi bahwa ...,22143,19003,1735693924,35,1,2,3,...,19003,Tiket ini di tutup Otomatis oleh sistem karena...,0c5fd7d70f7bcf1582ceddf831ded7a7,NaN,02/01/2025 15:13:58,0,01-01-2025,NaN,DPTSI,lupa password emailits permisi disini saya ing...
2,60022,Lupa pasword e-mail ITS,"<p>Selamat pagi, saya yang beridentitas dibawa...",18594,12467,1735697383,35,2,2,3,...,12467,Tiket ini di tutup Otomatis oleh sistem karena...,48ab9944afa5e054856536c286ac13de,NaN,03/01/2025 9:24:04,0,01-01-2025,NaN,DPTSI,lupa pasword e mail its selamat pagi saya yang...
3,60023,Tidak bisa login my ITS,<p>Selamat pagi perkenalkan saya mahasiswa bar...,21357,12467,1735697743,35,1,2,3,...,12467,Tiket ini di tutup Otomatis oleh sistem karena...,33646e6dffefaff780ca32a3142b9500,NaN,03/01/2025 9:03:09,0,01-01-2025,NaN,DPTSI,tidak bisa login my its selamat pagi perkenalk...
4,60024,Tidak bisa login di myits,"<p>saya ingin mereset password gmail saya, dik...",22144,12467,1735699517,35,1,2,3,...,12467,Tiket ini di tutup Otomatis oleh sistem karena...,06ea348c1ba74236b61dd3a7857995ed,NaN,03/01/2025 9:02:45,0,01-01-2025,NaN,DPTSI,tidak bisa login di myits saya ingin mereset p...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4676,65711,Akun myits tidak bisa terutentikasi,<p>aplikasi authenticator tidak memunculkan ko...,23870,19002,1750226938,35,1,1,3,...,19002,NaN,f6a3032bd547ce792011407918469f95,NaN,18/06/2025 13:25:11,0,18-06-2025,NaN,DPTSI,akun myits tidak bisa terutentikasi aplikasi a...
4677,65714,"Tidak bisa login myits, karena apk authenticat...","<p>Tidak bisa login myits, karena apk authenti...",23872,0,1750230015,35,2,0,3,...,0,NaN,ec1bba48d97fc4ab24897c1d92f6b0f1,NaN,18/06/2025 14:00:15,0,18-06-2025,NaN,DPTSI,tidak bisa login myits karena apk authenticato...
4678,65715,Lupa password integra,<p>Mohon bantuannya saya mahasiswa semester ak...,23871,0,1750230033,35,2,0,3,...,0,NaN,c7edc03d4d15095a6b0a12f1bc1c6428,NaN,18/06/2025 14:00:33,0,18-06-2025,NaN,DPTSI,lupa password integra mohon bantuannya saya ma...
4679,65717,Ganti Nomor Telpon MyIts,<p>Nomor telpon myITS saya sudah tidak aktif d...,21863,0,1750230802,22,2,0,3,...,0,NaN,a124f616e630ce84600e5566957209fb,NaN,18/06/2025 14:13:22,0,18-06-2025,NaN,DPTSI,ganti nomor telpon myits nomor telpon myits sa...


In [69]:
df_kb_label_3[['predicted_unit', 'score', 'confidence', 'keywords', 'all_unit_scores', 'all_unit_keywords']] = df_kb_label_3.apply(
    lambda row: implement_kb(row, kb_iteration_3), axis=1
)

In [70]:
df_kb_label_3['predicted_unit'].value_counts()

predicted_unit
DPTSI           2993
DPSP             455
DSDMO            325
DITMAWA          218
UKP              213
DIRPAIP          129
UNCLASSIFIED     113
BK                77
UNKNOWN           62
DRPM              35
BUK4L             26
BMA               22
KPM               13
Name: count, dtype: int64

In [77]:
df_kb_label_3.to_excel("../data/output/20260427_kb_implement_match_keyword_iteration_3.xlsx", index=False)

Iteration with no Unclassified

In [26]:
kb_iteration_3 = load_kb("../utils/kb_20260427_iteration_3.json")

In [27]:
df_kb_label_3_001 = df_kb_label.copy()

In [28]:
df_kb_label_3_001[['predicted_unit', 'score', 'confidence', 'keywords', 'all_unit_scores', 'all_unit_keywords']] = df_kb_label_3_001.apply(
    lambda row: implement_kb(row, kb_iteration_3), axis=1
)

In [29]:
df_kb_label_3_001['predicted_unit'].value_counts()

predicted_unit
DPTSI      2905
DPSP        483
DSDMO       351
DITMAWA     242
UKP         219
DIRPAIP     157
BK           93
DRPM         92
UNKNOWN      63
BMA          32
BUK4L        30
KPM          14
Name: count, dtype: int64

In [31]:
df_kb_label_3_001.to_excel("../data/output/20260429_kb_implement_match_keyword_iteration_3_001.xlsx", index=False)

### Iteration 4

In [36]:
kb_iteration_4 = load_kb("../utils/json_kb/kb_20260429_iteration_4.json")

In [39]:
df_kb_label_4 = df_kb_label.copy()

In [40]:
df_kb_label_4[['predicted_unit', 'score', 'confidence', 'keywords', 'all_unit_scores', 'all_unit_keywords']] = df_kb_label_4.apply(
    lambda row: implement_kb(row, kb_iteration_4), axis=1
)
df_kb_label_4['predicted_unit'].value_counts()

predicted_unit
DPTSI      2792
DPSP        454
DSDMO       352
DITMAWA     219
UKP         218
DIRPAIP     150
BK           92
DRPM         63
UNKNOWN      38
BMA          32
BUK4L        27
KPM          13
Name: count, dtype: int64

In [41]:
df_kb_label_4.to_excel("../data/output/20260502_kb_implement_match_keyword_iteration_4_002.xlsx", index=False)